In [4]:
from google.colab import drive
drive.mount('/content/drive')

# cd to the *root* of career-ml, NOT to scripts
%cd "/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml


In [6]:
from pathlib import Path
import re
from typing import List, Dict, Tuple

import pandas as pd

# -------- paths --------
try:
    BASE_DIR = Path(__file__).resolve().parents[1]
except NameError:
    BASE_DIR = Path.cwd()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SKILLS_PATH = PROCESSED_DIR / "skills_cleaned.csv"


# -------- helpers --------

def normalize_text(text: str) -> str:
    """Lowercase, remove weird chars, collapse spaces."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # keep letters, digits, #, +, ., and separators
    text = re.sub(r"[^a-z0-9#+./,; ]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def build_skill_patterns() -> List[Dict]:
    """Load skills_cleaned.csv and build patterns for matching."""
    if not SKILLS_PATH.exists():
        raise FileNotFoundError(f"{SKILLS_PATH} not found")

    df = pd.read_csv(SKILLS_PATH)
    skills = []

    for _, row in df.iterrows():
        skill_id = str(row["skill_id"]).strip()
        name = str(row["name"]).strip()
        aliases_raw = str(row.get("aliases", "")).strip()

        phrases = [name]
        if aliases_raw:
            phrases.extend(a.strip() for a in aliases_raw.split(";") if a.strip())

        norm_phrases = sorted(set(normalize_text(p) for p in phrases if p))
        norm_phrases = [p for p in norm_phrases if p and len(p) > 1]
        if not norm_phrases:
            continue

        skills.append(
            {
                "skill_id": skill_id,
                "name": name,
                "patterns": norm_phrases,
            }
        )

    print(f"Loaded {len(skills)} skills from {SKILLS_PATH}")
    return skills


def match_skills_in_text(text: str, skills: List[Dict]) -> Tuple[List[str], List[str]]:
    """Given a text and a list of skills, return matching skill_ids and names."""
    norm = normalize_text(text)
    if not norm:
        return [], []

    padded = f" {norm} "

    matched_ids = []
    matched_names = []

    for s in skills:
        skill_id = s["skill_id"]
        name = s["name"]
        for pat in s["patterns"]:
            pat_padded = f" {pat} "
            if pat_padded in padded:
                matched_ids.append(skill_id)
                matched_names.append(name)
                break  # avoid duplicate matches for same skill

    # de-duplicate, preserve order
    seen = set()
    ids_unique = []
    names_unique = []
    for sid, nm in zip(matched_ids, matched_names):
        if sid not in seen:
            seen.add(sid)
            ids_unique.append(sid)
            names_unique.append(nm)

    return ids_unique, names_unique


def tag_dataframe(
    df: pd.DataFrame,
    text_columns: List[str],
    skills: List[Dict],
    id_prefix: str,
    source_name: str,
) -> pd.DataFrame:
    """
    For each row in df, combine the chosen text columns and match skills.
    Returns a new DataFrame with extra tagging columns + 'source' + 'job_uid'.
    """
    print(f"\nTagging dataset from source '{source_name}' ...")

    df = df.copy()

    # add source column
    df["source"] = source_name

    # create synthetic unique ID if not present
    if "job_uid" not in df.columns:
        df["job_uid"] = [f"{id_prefix}_{i+1:06d}" for i in range(len(df))]

    def row_text(row):
        parts = []
        for col in text_columns:
            if col in df.columns:
                parts.append(str(row[col]))
        return " ".join(parts)

    matched_ids_all = []
    matched_names_all = []
    counts = []

    for _, row in df.iterrows():
        txt = row_text(row)
        ids, names = match_skills_in_text(txt, skills)
        matched_ids_all.append(";".join(ids))
        matched_names_all.append(";".join(names))
        counts.append(len(ids))

    df["matched_skill_ids"] = matched_ids_all
    df["matched_skills"] = matched_names_all
    df["matched_skill_count"] = counts

    print(
        f"Tagged {len(df)} rows from '{source_name}'. "
        f"Average matched skills: {sum(counts)/len(counts):.2f}"
    )
    return df


# -------- main pipeline --------

def main():
    skills = build_skill_patterns()
    tagged_frames = []

    # 1) jobs.csv
    jobs_path = RAW_DIR / "jobs.csv"
    if jobs_path.exists():
        jobs_df = pd.read_csv(jobs_path, encoding="utf-8", low_memory=False)
        jobs_tagged = tag_dataframe(
            jobs_df,
            text_columns=["taglist", "description", "job_name"],
            skills=skills,
            id_prefix="JOBS",
            source_name="jobs.csv",
        )
        tagged_frames.append(jobs_tagged)
    else:
        print("WARNING: jobs.csv not found in data/raw")

    # 2) Dataset_jotpars.csv
    ds_path = RAW_DIR / "Dataset_jotpars.csv"
    if ds_path.exists():
        ds_df = pd.read_csv(ds_path, encoding="latin1", low_memory=False)
        ds_tagged = tag_dataframe(
            ds_df,
            text_columns=["requirment", "description", "name"],
            skills=skills,
            id_prefix="JPARS",
            source_name="Dataset_jotpars.csv",
        )
        tagged_frames.append(ds_tagged)
    else:
        print("WARNING: Dataset_jotpars.csv not found in data/raw")

    # 3) CleanData_jotpars.csv
    clean_path = RAW_DIR / "CleanData_jotpars.csv"
    if clean_path.exists():
        clean_df = pd.read_csv(clean_path, encoding="latin1", low_memory=False)
        clean_tagged = tag_dataframe(
            clean_df,
            text_columns=["r", "description", "title"],
            skills=skills,
            id_prefix="CLEAN",
            source_name="CleanData_jotpars.csv",
        )
        tagged_frames.append(clean_tagged)
    else:
        print("WARNING: CleanData_jotpars.csv not found in data/raw")

    # 4) Job opportunities.xlsx
    jobops_path = RAW_DIR / "Job opportunities.xlsx"
    if jobops_path.exists():
        jobops_df = pd.read_excel(jobops_path, engine="openpyxl")
        jobops_tagged = tag_dataframe(
            jobops_df,
            text_columns=["Required Skills", "Job Description"],
            skills=skills,
            id_prefix="JOBSUK",
            source_name="Job opportunities.xlsx",
        )
        tagged_frames.append(jobops_tagged)
    else:
        print("WARNING: Job opportunities.xlsx not found in data/raw")

    if not tagged_frames:
        print("ERROR: No datasets found to tag. Check data/raw.")
        return

    # ---- combine everything into ONE dataset ----
    combined = pd.concat(tagged_frames, ignore_index=True)
    out_path = PROCESSED_DIR / "all_jobs_tagged.csv"
    combined.to_csv(out_path, index=False)
    print(f"\nSaved combined tagged dataset with {len(combined)} rows to {out_path}")


if __name__ == "__main__":
    main()


Loaded 284 skills from /content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml/data/processed/skills_cleaned.csv

Tagging dataset from source 'jobs.csv' ...
Tagged 1412 rows from 'jobs.csv'. Average matched skills: 9.23

Tagging dataset from source 'Dataset_jotpars.csv' ...
Tagged 8797 rows from 'Dataset_jotpars.csv'. Average matched skills: 12.27

Tagging dataset from source 'CleanData_jotpars.csv' ...
Tagged 4657 rows from 'CleanData_jotpars.csv'. Average matched skills: 11.91

Tagging dataset from source 'Job opportunities.xlsx' ...
Tagged 364 rows from 'Job opportunities.xlsx'. Average matched skills: 1.23

Saved combined tagged dataset with 15230 rows to /content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml/data/processed/all_jobs_tagged.csv
